In [4]:
from pathlib import Path

# Current notebook is inside: supply-chain-control-tower/notebooks
# Go up one level to the actual project folder
PROJECT_ROOT = Path.cwd().parent
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

required_files = [
    "dashboard_shipping_mode_performance.csv",
    "dashboard_regional_delivery_risk.csv",
    "dashboard_forecast_model_comparison.csv",
    "dashboard_inventory_recommendations.csv",
    "dashboard_daily_forecast_predictions.csv",
    "dashboard_executive_kpis.csv",
]

missing_files = [file for file in required_files if not (OUTPUTS_DIR / file).exists()]

print("Project folder:", PROJECT_ROOT)
print("Outputs folder:", OUTPUTS_DIR)

if missing_files:
    print("\nMissing files:")
    for file in missing_files:
        print(" -", file)
else:
    print("\nAll required AI Copilot data files are available.")

Project folder: /Users/nikhilkoushik/Desktop/supply-chain-control-tower
Outputs folder: /Users/nikhilkoushik/Desktop/supply-chain-control-tower/outputs

All required AI Copilot data files are available.


In [5]:
from pathlib import Path
from datetime import datetime
import duckdb
import pandas as pd

# Create a database folder inside the project
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

DB_PATH = DATA_DIR / "supply_chain_copilot.duckdb"

# Tableau-ready files that the AI Copilot is allowed to query
table_files = {
    "shipping_mode_performance": "dashboard_shipping_mode_performance.csv",
    "regional_delivery_risk": "dashboard_regional_delivery_risk.csv",
    "forecast_model_comparison": "dashboard_forecast_model_comparison.csv",
    "inventory_recommendations": "dashboard_inventory_recommendations.csv",
    "daily_forecast_predictions": "dashboard_daily_forecast_predictions.csv",
    "executive_kpis": "dashboard_executive_kpis.csv",
}

# Connect to DuckDB database
con = duckdb.connect(str(DB_PATH))

# Load each approved CSV into its own table
for table_name, file_name in table_files.items():
    file_path = OUTPUTS_DIR / file_name
    df = pd.read_csv(file_path)

    con.register("temp_dataframe", df)
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM temp_dataframe
    """)
    con.unregister("temp_dataframe")

# Record refresh details for the automated pipeline
con.execute("""
    CREATE TABLE IF NOT EXISTS refresh_history (
        refresh_timestamp TIMESTAMP,
        refresh_status VARCHAR,
        source VARCHAR
    )
""")

con.execute(
    "INSERT INTO refresh_history VALUES (?, ?, ?)",
    [datetime.now(), "SUCCESS", "Initial AI Copilot database build"]
)

# Verify tables and row counts
table_summary = []

for table_name in table_files:
    row_count = con.execute(
        f"SELECT COUNT(*) FROM {table_name}"
    ).fetchone()[0]

    table_summary.append(
        {"table_name": table_name, "row_count": row_count}
    )

con.close()

display(pd.DataFrame(table_summary))
print(f"\nAI Copilot database created successfully:\n{DB_PATH}")

,table_name,row_count
0,shipping_mode_performance,4
1,regional_delivery_risk,23
2,forecast_model_comparison,3
3,inventory_recommendations,20
4,daily_forecast_predictions,560
5,executive_kpis,10



AI Copilot database created successfully:
/Users/nikhilkoushik/Desktop/supply-chain-control-tower/data/supply_chain_copilot.duckdb


In [6]:
%pip install streamlit plotly duckdb -q

Note: you may need to restart the kernel to use updated packages.


In [12]:
%%writefile ../app.py
from pathlib import Path
import re

import duckdb
import pandas as pd
import plotly.express as px
import streamlit as st


# -----------------------------
# App configuration
# -----------------------------
st.set_page_config(
    page_title="AI Supply Chain Copilot",
    page_icon="📦",
    layout="wide",
)

PROJECT_ROOT = Path(__file__).resolve().parent
DB_PATH = PROJECT_ROOT / "data" / "supply_chain_copilot.duckdb"


# -----------------------------
# Database helpers
# -----------------------------
def clean_column_name(column_name):
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    return column_name.strip("_")


@st.cache_data
def load_table(table_name):
    with duckdb.connect(str(DB_PATH), read_only=True) as con:
        dataframe = con.execute(f"SELECT * FROM {table_name}").df()

    dataframe.columns = [clean_column_name(column) for column in dataframe.columns]
    return dataframe


def first_matching_column(dataframe, candidates):
    for column in candidates:
        if column in dataframe.columns:
            return column
    return None


def format_currency(value):
    if pd.isna(value):
        return "N/A"
    if abs(value) >= 1_000_000:
        return f"${value / 1_000_000:.2f}M"
    if abs(value) >= 1_000:
        return f"${value / 1_000:.0f}K"
    return f"${value:,.0f}"


# -----------------------------
# Load approved analytics tables
# -----------------------------
shipping = load_table("shipping_mode_performance")
regional = load_table("regional_delivery_risk")
models = load_table("forecast_model_comparison")
inventory = load_table("inventory_recommendations")
daily_forecast = load_table("daily_forecast_predictions")
kpis = load_table("executive_kpis")

shipping_mode_col = first_matching_column(shipping, ["shipping_mode"])
shipping_late_rate_col = first_matching_column(shipping, ["late_delivery_rate"])

region_col = first_matching_column(regional, ["order_region", "region"])
late_shipments_col = first_matching_column(
    regional,
    ["late_shipment_records", "late_delivery_records"]
)
region_late_rate_col = first_matching_column(regional, ["late_delivery_rate"])

model_name_col = first_matching_column(models, ["model"])
wape_col = first_matching_column(models, ["wape_percent", "wape"])

item_col = first_matching_column(inventory, ["item_id"])
inventory_category_col = first_matching_column(inventory, ["cat_id", "category"])
recommended_stock_col = first_matching_column(
    inventory,
    ["recommended_28_day_stock_units", "recommended_28_day_stock"]
)
reorder_point_col = first_matching_column(inventory, ["reorder_point_units", "reorder_point"])
safety_stock_col = first_matching_column(inventory, ["safety_stock_units", "safety_stock"])
revenue_col = first_matching_column(inventory, ["total_revenue"])

daily_item_col = first_matching_column(daily_forecast, ["item_id"])
date_col = first_matching_column(daily_forecast, ["date"])
actual_col = first_matching_column(daily_forecast, ["actual_units_sold", "actual_units"])
prediction_col = first_matching_column(
    daily_forecast,
    ["lightgbm_recursive_prediction", "recursive_lightgbm_prediction"]
)

# -----------------------------
# Header
# -----------------------------
st.title("📦 AI Supply Chain Copilot")
st.caption(
    "Delivery-risk diagnostics, 28-day demand forecasting, and inventory replenishment recommendations."
)

# -----------------------------
# Executive metrics
# -----------------------------
best_model = models.loc[models[wape_col].idxmin()]
best_wape = float(best_model[wape_col])
forecast_accuracy = 100 - best_wape

sales_value = 33_054_402.38
late_delivery_rate = 54.83

metric_1, metric_2, metric_3, metric_4 = st.columns(4)

metric_1.metric("Total Sales", format_currency(sales_value))
metric_2.metric("Late-Delivery Rate", f"{late_delivery_rate:.2f}%")
metric_3.metric("Best Forecast WAPE", f"{best_wape:.2f}%")
metric_4.metric("WAPE-Derived Accuracy", f"{forecast_accuracy:.2f}%")

st.divider()

# -----------------------------
# Dashboard tabs
# -----------------------------
delivery_tab, forecast_tab, inventory_tab = st.tabs(
    ["Delivery Risk", "Demand Forecast", "Inventory Recommendations"]
)

with delivery_tab:
    left_column, right_column = st.columns(2)

    with left_column:
        st.subheader("Late Delivery Rate by Shipping Mode")

        shipping_chart = shipping.sort_values(
            shipping_late_rate_col,
            ascending=False
        )

        figure = px.bar(
            shipping_chart,
            x=shipping_late_rate_col,
            y=shipping_mode_col,
            orientation="h",
            color=shipping_late_rate_col,
            color_continuous_scale="Reds",
            text_auto=".2f",
            labels={
                shipping_late_rate_col: "Late Delivery Rate (%)",
                shipping_mode_col: "Shipping Mode",
            },
        )

        figure.update_layout(
            yaxis={"categoryorder": "total ascending"},
            coloraxis_showscale=False,
            height=400,
        )

        st.plotly_chart(figure, use_container_width=True)

    with right_column:
        st.subheader("Top Regional Delivery-Risk Priorities")

        regional_chart = regional.sort_values(
            late_shipments_col,
            ascending=False
        ).head(10)

        figure = px.bar(
            regional_chart,
            x=late_shipments_col,
            y=region_col,
            orientation="h",
            color=region_late_rate_col,
            color_continuous_scale="Oranges",
            text_auto=",",
            labels={
                late_shipments_col: "Late Shipment Records",
                region_col: "Order Region",
                region_late_rate_col: "Late Delivery Rate (%)",
            },
        )

        figure.update_layout(
            yaxis={"categoryorder": "total ascending"},
            height=400,
        )

        st.plotly_chart(figure, use_container_width=True)

with forecast_tab:
    left_column, right_column = st.columns([1, 2])

    with left_column:
        st.subheader("Model Comparison")

        model_chart = models.sort_values(wape_col, ascending=True)

        model_colors = {
            "Recursive LightGBM": "#2ca02c",
            "Initial ML Model": "#9e9e9e",
            "Seasonal Naïve Baseline": "#ef476f",
        }

        figure = px.bar(
            model_chart,
            x=wape_col,
            y=model_name_col,
            orientation="h",
            color=model_name_col,
            color_discrete_map=model_colors,
            text_auto=".2f",
            labels={
                wape_col: "WAPE (%)",
                model_name_col: "Model",
            },
        )

        figure.update_layout(
            showlegend=False,
            yaxis={"categoryorder": "total ascending"},
            height=400,
        )

        st.plotly_chart(figure, use_container_width=True)

    with right_column:
        st.subheader("Actual vs Recursive LightGBM Forecast")

        selected_sku = st.selectbox(
            "Select SKU",
            sorted(daily_forecast[daily_item_col].unique())
        )

        sku_data = daily_forecast[
            daily_forecast[daily_item_col] == selected_sku
        ].copy()

        sku_data[date_col] = pd.to_datetime(sku_data[date_col])
        sku_data = sku_data.sort_values(date_col)

        chart_data = sku_data.melt(
            id_vars=[date_col],
            value_vars=[actual_col, prediction_col],
            var_name="Series",
            value_name="Units Sold",
        )

        chart_data["Series"] = chart_data["Series"].replace({
            actual_col: "Actual Units Sold",
            prediction_col: "Recursive LightGBM Forecast",
        })

        figure = px.line(
            chart_data,
            x=date_col,
            y="Units Sold",
            color="Series",
            color_discrete_map={
                "Actual Units Sold": "#1f77b4",
                "Recursive LightGBM Forecast": "#ef476f",
            },
            markers=True,
        )

        figure.update_layout(height=400)
        st.plotly_chart(figure, use_container_width=True)

with inventory_tab:
    st.subheader("Bias-Adjusted Inventory Replenishment Recommendations")

    categories = ["All"] + sorted(inventory[inventory_category_col].dropna().unique())

    selected_category = st.selectbox(
        "Filter by Category",
        categories
    )

    inventory_view = inventory.copy()

    if selected_category != "All":
        inventory_view = inventory_view[
            inventory_view[inventory_category_col] == selected_category
        ]

    inventory_view = inventory_view.sort_values(
        recommended_stock_col,
        ascending=False
    )

    display_columns = [
        item_col,
        inventory_category_col,
        revenue_col,
        recommended_stock_col,
        reorder_point_col,
        safety_stock_col,
    ]

    inventory_view = inventory_view[display_columns].rename(columns={
        item_col: "SKU",
        inventory_category_col: "Category",
        revenue_col: "Total Revenue",
        recommended_stock_col: "Recommended 28-Day Stock",
        reorder_point_col: "Reorder Point",
        safety_stock_col: "Safety Stock",
    })

    st.dataframe(
        inventory_view,
        use_container_width=True,
        hide_index=True,
    )

st.divider()
st.caption(
    "Data source: validated project outputs. Forecast: Recursive LightGBM, "
    "28-day horizon. Inventory recommendations use bias adjustment and safety stock."
)

Overwriting ../app.py


In [10]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
secrets_path = PROJECT_ROOT / ".streamlit" / "secrets.toml"

secrets_path.parent.mkdir(exist_ok=True)

if not secrets_path.exists():
    secrets_path.write_text('GEMINI_API_KEY = ""\n')

print(secrets_path)

/Users/nikhilkoushik/Desktop/supply-chain-control-tower/.streamlit/secrets.toml


In [11]:
%pip install -q google-genai

Note: you may need to restart the kernel to use updated packages.


In [15]:
%%writefile ../copilot_chat.py

from pathlib import Path
import re
import duckdb
import pandas as pd
from google import genai


PROJECT_ROOT = Path(__file__).resolve().parent
DB_PATH = PROJECT_ROOT / "data" / "supply_chain_copilot.duckdb"


def read_table(table_name):
    with duckdb.connect(str(DB_PATH), read_only=True) as con:
        return con.execute(f"SELECT * FROM {table_name}").df()


def find_sku(question):
    match = re.search(r"(FOODS|HOBBIES|HOUSEHOLD)_\d_\d{3}", question.upper())
    return match.group(0) if match else None


def get_evidence(question):
    q = question.lower()
    sku = find_sku(question)

    # Inventory questions
    if any(word in q for word in ["inventory", "stock", "reorder", "safety stock", "replenish"]):
        df = read_table("inventory_recommendations")

        if sku:
            return df[df["item_id"].str.upper() == sku].copy(), "Inventory recommendation"

        if "food" in q:
            df = df[df["cat_id"].str.upper() == "FOODS"]
        elif "household" in q:
            df = df[df["cat_id"].str.upper() == "HOUSEHOLD"]
        elif "hobbies" in q or "hobby" in q:
            df = df[df["cat_id"].str.upper() == "HOBBIES"]

        sort_column = "recommended_28_day_stock_units"
        return df.sort_values(sort_column, ascending=False).head(10), "Top inventory recommendations"

    # SKU-specific forecast questions
    if sku or any(word in q for word in ["forecast", "demand", "predict", "prediction"]):
        df = read_table("daily_forecast_predictions")

        if sku:
            df = df[df["item_id"].str.upper() == sku].copy()
        else:
            df = (
                df.groupby("item_id", as_index=False)
                .agg(
                    actual_units_sold=("actual_units_sold", "sum"),
                    lightgbm_recursive_prediction=("lightgbm_recursive_prediction", "sum")
                )
                .sort_values("lightgbm_recursive_prediction", ascending=False)
                .head(10)
            )

        return df, "Recursive LightGBM forecast evidence"

    # Shipping-mode questions
    if any(word in q for word in ["shipping mode", "first class", "second class", "same day", "standard class"]):
        df = read_table("shipping_mode_performance")
        return df.sort_values("late_delivery_rate", ascending=False), "Shipping-mode performance"

    # Regional delivery-risk questions
    if any(word in q for word in ["region", "regional", "central america", "western europe", "delivery risk"]):
        df = read_table("regional_delivery_risk")

        regions = df["order_region"].astype(str)
        matches = df[regions.str.lower().apply(lambda x: x in q)]

        if not matches.empty:
            return matches, "Regional delivery-risk evidence"

        return (
            df.sort_values("priority_score", ascending=False).head(10),
            "Top regional delivery-risk priorities"
        )

    # Model comparison questions
    if any(word in q for word in ["model", "wape", "accuracy", "mae", "rmse", "bias", "baseline", "lightgbm"]):
        df = read_table("forecast_model_comparison")
        return df, "Forecast model comparison"

    # Executive KPI default
    return read_table("executive_kpis"), "Executive KPI evidence"


def answer_question(question, api_key):
    evidence, evidence_title = get_evidence(question)

    # Safety: only the selected, approved evidence is sent to the model.
    evidence_text = evidence.head(20).to_csv(index=False)

    if not api_key:
        return (
            "The AI key is missing. Add GEMINI_API_KEY to .streamlit/secrets.toml, "
            "save it, and restart Streamlit."
        ), evidence, evidence_title

    client = genai.Client(api_key=api_key)

    prompt = f"""
You are the AI Supply Chain Copilot for a portfolio project.

Answer the user's question using ONLY the supplied evidence.
Do not invent data, assume live data, or claim causal proof beyond the evidence.
The source is historical project data and forecast outputs. Keep the answer concise,
business-focused, and include key numbers when available.

User question:
{question}

Evidence title:
{evidence_title}

Evidence:
{evidence_text}
"""

    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )
        return response.text, evidence, evidence_title

    except Exception as error:
        return (
            f"The Copilot could not generate an AI response. "
            f"Check the API key and internet connection. Technical detail: {error}"
        ), evidence, evidence_title

Overwriting ../copilot_chat.py


In [16]:
%%writefile ../requirements.txt
streamlit
pandas
plotly
duckdb
google-genai

Writing ../requirements.txt


In [17]:
%%writefile -a ../.gitignore
.streamlit/secrets.toml
__pycache__/
*.pyc
.ipynb_checkpoints/
.DS_Store

Writing ../.gitignore
